# Этап 2: Деноизинг корпуса (Google Colab + GPU)

**Что делает этот ноутбук:**
1. Устанавливает DeepFilterNet3 (лучшая открытая модель деноизинга речи)
2. Обрабатывает весь корпус батчами с автосохранением прогресса
3. После деноизинга пересчитывает SNR и применяет финальные фильтры
4. Генерирует `manifest_clean.csv` — готовый к обучению StarGANv2-VC

**Требования:** Colab Pro с GPU (A100/V100/T4)

**Время:** ~40–60 мин на весь корпус (A100)

In [ ]:
# ── Шаг 0: Монтируем Drive ───────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os

# ════════════════════════════════════════════════════════════
# НАСТРОЙ ПУТИ ПОД СВОЮ СТРУКТУРУ
# ════════════════════════════════════════════════════════════
DRIVE_BASE   = '/content/drive/MyDrive/diploma'
CORPUS_DIR   = f'{DRIVE_BASE}/corpus_24k'          # папка с аудио (speaker_id/file.wav)
MANIFEST_IN  = f'{CORPUS_DIR}/manifest_filtered.csv'  # выход step1
OUT_DIR      = f'{DRIVE_BASE}/corpus_clean'        # куда писать очищенные файлы
MANIFEST_OUT = f'{OUT_DIR}/manifest_clean.csv'     # финальный манифест
PROGRESS_FILE = f'{DRIVE_BASE}/denoise_progress.json'  # файл прогресса
# ════════════════════════════════════════════════════════════

os.makedirs(OUT_DIR, exist_ok=True)
print(f'✅ Drive смонтирован')
print(f'   Корпус:    {CORPUS_DIR}')
print(f'   Выход:     {OUT_DIR}')

In [ ]:
# ── Шаг 1: Установка зависимостей ───────────────────────────
# DeepFilterNet3 — state-of-the-art модель деноизинга (2023)
# Значительно лучше DFN2 на записях с комнатным шумом и реверберацией
!pip install -q deepfilternet==0.5.6
!pip install -q soundfile librosa pandas numpy tqdm

# Проверяем GPU
import torch
print(f'\nGPU доступен: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'Модель: {torch.cuda.get_device_name(0)}')
    print(f'Память: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# ── Шаг 2: Импорты и инициализация DeepFilterNet3 ────────────
import json
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import soundfile as sf
import librosa
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')

# Инициализируем DeepFilterNet3
from df.enhance import enhance, init_df
from df.io import load_audio as df_load, save_audio as df_save

# init_df() автоматически выбирает DFN3 если установлен deepfilternet >= 0.5
model, df_state, _ = init_df()
print(f'✅ DeepFilterNet инициализирован')
print(f'   Целевой SR модели: {df_state.sr()} Гц')

TARGET_SR = 24_000   # StarGANv2-VC требует 24кГц
TARGET_RMS_DB = -20.0  # целевой уровень громкости

In [ ]:
# ── Шаг 3: Вспомогательные функции ──────────────────────────

def compute_snr_db(y: np.ndarray, frame_len: int = 2048) -> float:
    """
    Оценка SNR через сравнение громких и тихих фреймов.
    Сигнал = медиана 25% громких фреймов.
    Шум    = медиана 25% тихих фреймов.
    """
    if len(y) < frame_len * 2:
        return float(np.clip(20 * np.log10(np.sqrt(np.mean(y**2)) + 1e-10), 0, 60))
    
    frames = librosa.util.frame(y, frame_length=frame_len, hop_length=frame_len//2)
    rms_frames = np.sqrt(np.mean(frames**2, axis=0))
    
    n = len(rms_frames)
    sorted_rms = np.sort(rms_frames)
    
    signal_level = np.median(sorted_rms[int(0.75*n):]) + 1e-10
    noise_level  = np.median(sorted_rms[:max(1, int(0.25*n))]) + 1e-10
    
    snr = 20 * np.log10(signal_level / noise_level)
    return float(np.clip(snr, 0, 60))


def normalize_rms(y: np.ndarray, target_db: float = -20.0) -> np.ndarray:
    """RMS-нормализация с защитой от клиппирования."""
    rms = np.sqrt(np.mean(y**2))
    if rms < 1e-6:
        return y
    gain = 10 ** ((target_db - 20 * np.log10(rms)) / 20)
    y_out = y * gain
    peak = np.max(np.abs(y_out))
    if peak > 0.99:
        y_out *= 0.99 / peak
    return y_out


def trim_silence(y: np.ndarray, sr: int,
                 top_db: float = 35.0, pad_ms: int = 50) -> np.ndarray:
    """VAD-обрезка тишины с сохранением небольшого отступа."""
    y_trimmed, _ = librosa.effects.trim(y, top_db=top_db)
    pad = np.zeros(int(sr * pad_ms / 1000), dtype=y.dtype)
    return np.concatenate([pad, y_trimmed, pad])


def segment_by_silence(y: np.ndarray, sr: int,
                        max_dur: float = 10.0,
                        min_dur: float = 1.5) -> list:
    """
    Разбивает длинную запись на сегменты по паузам.
    Возвращает список numpy-массивов.
    """
    duration = len(y) / sr
    if duration <= max_dur:
        return [y] if duration >= min_dur else []
    
    intervals = librosa.effects.split(y, top_db=35,
                                       frame_length=2048, hop_length=512)
    segments = []
    current_start = intervals[0][0] if len(intervals) > 0 else 0
    current_end   = intervals[0][1] if len(intervals) > 0 else len(y)
    
    for start, end in intervals[1:]:
        seg_dur = (end - current_start) / sr
        gap     = (start - current_end) / sr
        
        if seg_dur > max_dur or gap > 0.5:
            # Завершаем текущий сегмент
            seg = y[current_start:current_end]
            if len(seg) / sr >= min_dur:
                segments.append(seg)
            current_start = start
        
        current_end = end
    
    # Последний сегмент
    seg = y[current_start:current_end]
    if len(seg) / sr >= min_dur:
        segments.append(seg)
    
    return segments if segments else ([y] if duration >= min_dur else [])


def denoise_with_dfn(y: np.ndarray, sr: int,
                      model, df_state) -> np.ndarray:
    """
    Деноизинг через DeepFilterNet3.
    DFN работает на 48кГц внутри, поэтому:
    1. Ресемплируем 24к → 48к
    2. Деноизируем
    3. Ресемплируем 48к → 24к
    """
    dfn_sr = df_state.sr()  # обычно 48000
    
    # Ресемплируем на частоту DFN
    y_up = librosa.resample(y, orig_sr=sr, target_sr=dfn_sr,
                             res_type='soxr_hq')
    
    # Деноизируем
    audio_tensor = torch.from_numpy(y_up).float().unsqueeze(0)
    enhanced = enhance(model, df_state, audio_tensor)
    y_enhanced = enhanced.squeeze(0).numpy()
    
    # Ресемплируем обратно
    y_out = librosa.resample(y_enhanced, orig_sr=dfn_sr, target_sr=sr,
                              res_type='soxr_hq')
    return y_out


print('✅ Вспомогательные функции загружены')

In [ ]:
# ── Шаг 4: Загрузка манифеста и инициализация прогресса ─────

df = pd.read_csv(MANIFEST_IN)
print(f'Загружено {len(df)} записей из {MANIFEST_IN}')

# Загружаем прогресс (для продолжения после обрыва сессии)
if os.path.exists(PROGRESS_FILE):
    with open(PROGRESS_FILE, 'r') as f:
        progress = json.load(f)
    already_done = set(progress.get('done_ids', []))
    all_records = progress.get('records', [])
    print(f'🔄 Продолжаем: уже обработано {len(already_done)} записей')
else:
    already_done = set()
    all_records = []
    print('🆕 Начинаем с нуля')

# Записи для обработки
df_todo = df[~df['utterance_id'].isin(already_done)]
print(f'Осталось обработать: {len(df_todo)} записей')

In [ ]:
# ── Шаг 5: ОСНОВНОЙ ЦИКЛ ОБРАБОТКИ ─────────────────────────
# Параметры
SAVE_CHECKPOINT_EVERY = 500   # сохраняем прогресс каждые N записей
MIN_SNR_AFTER_DENOISE = 10.0  # финальный фильтр после деноизинга
MIN_DUR_AFTER_SEG     = 1.5   # мин. длит. сегмента после разбивки

errors = 0
skipped_low_snr = 0
start_time = time.time()

for batch_start in range(0, len(df_todo), SAVE_CHECKPOINT_EVERY):
    batch = df_todo.iloc[batch_start:batch_start + SAVE_CHECKPOINT_EVERY]
    
    for _, row in tqdm(batch.iterrows(), total=len(batch),
                        desc=f'Батч {batch_start//SAVE_CHECKPOINT_EVERY + 1}'):
        utt_id = row['utterance_id']
        
        # Путь к исходному файлу
        src_path = Path(CORPUS_DIR) / row['rel_path']
        if not src_path.exists():
            errors += 1
            continue
        
        try:
            # ── Загрузка ──────────────────────────────────────
            y, orig_sr = sf.read(str(src_path), dtype='float32',
                                  always_2d=False)
            if y.ndim > 1:
                y = y.mean(axis=1)
            
            # ── Ресемплинг до 24кГц (если ещё не) ───────────
            if orig_sr != TARGET_SR:
                y = librosa.resample(y, orig_sr=orig_sr,
                                      target_sr=TARGET_SR,
                                      res_type='soxr_hq')
            
            # ── Деноизинг (для всех записей с SNR < 20 дБ) ──
            needs_denoise = row.get('needs_denoise', True)
            if needs_denoise:
                y = denoise_with_dfn(y, TARGET_SR, model, df_state)
            
            # ── VAD-обрезка ───────────────────────────────────
            y = trim_silence(y, TARGET_SR)
            
            # ── RMS-нормализация ──────────────────────────────
            y = normalize_rms(y, TARGET_RMS_DB)
            
            # ── Проверка SNR после деноизинга ─────────────────
            snr_after = compute_snr_db(y)
            if snr_after < MIN_SNR_AFTER_DENOISE:
                skipped_low_snr += 1
                already_done.add(utt_id)  # помечаем как обработанную
                continue  # отбрасываем — деноизинг не помог
            
            # ── Сегментация длинных записей ───────────────────
            segments = segment_by_silence(
                y, TARGET_SR,
                max_dur=10.0,
                min_dur=MIN_DUR_AFTER_SEG
            )
            
            if not segments:
                already_done.add(utt_id)
                continue
            
            # ── Сохранение ────────────────────────────────────
            speaker_id = row['speaker_id']
            out_spk_dir = Path(OUT_DIR) / speaker_id
            out_spk_dir.mkdir(parents=True, exist_ok=True)
            
            for seg_idx, seg in enumerate(segments):
                if len(segments) == 1:
                    new_id = utt_id
                else:
                    new_id = f'{utt_id}_seg{seg_idx:02d}'
                
                out_path = out_spk_dir / f'{new_id}.wav'
                sf.write(str(out_path), seg, TARGET_SR, subtype='PCM_16')
                
                seg_snr = compute_snr_db(seg)
                seg_rms = 20 * np.log10(np.sqrt(np.mean(seg**2)) + 1e-10)
                
                all_records.append({
                    'utterance_id':   new_id,
                    'rel_path':       f'{speaker_id}/{new_id}.wav',
                    'speaker_id':     speaker_id,
                    'domain_id':      row['domain_id'],
                    'dialect_group':  row['dialect_group'],
                    'gender':         row['gender'],
                    'birth_year':     row['birth_year'],
                    'duration_sec':   round(len(seg) / TARGET_SR, 4),
                    'snr_db':         round(seg_snr, 2),
                    'snr_db_original': round(float(row['snr_db']), 2),
                    'rms_db':         round(seg_rms, 2),
                    'was_denoised':   bool(needs_denoise),
                    'transcript':     row['transcript'],
                    'split':          row['split'],
                })
            
            already_done.add(utt_id)
        
        except Exception as e:
            errors += 1
            if errors <= 5:
                print(f'  [ERR] {utt_id}: {e}')
    
    # Сохраняем прогресс после каждого батча
    with open(PROGRESS_FILE, 'w') as f:
        json.dump({'done_ids': list(already_done),
                   'records': all_records}, f)
    
    elapsed = (time.time() - start_time) / 60
    total_done = len(already_done)
    speed = total_done / elapsed if elapsed > 0 else 0
    remaining = (len(df) - total_done) / speed if speed > 0 else 0
    print(f'  💾 Прогресс сохранён. Обработано: {total_done}/{len(df)} '
          f'| Прошло: {elapsed:.0f} мин | Осталось ~{remaining:.0f} мин')

print(f'\n✅ Обработка завершена!')
print(f'   Ошибок: {errors}')
print(f'   Отброшено (SNR после денойза < {MIN_SNR_AFTER_DENOISE} дБ): {skipped_low_snr}')
print(f'   Итоговых записей: {len(all_records)}')

In [ ]:
# ── Шаг 6: Сохранение финального манифеста ──────────────────

df_clean = pd.DataFrame(all_records)

# Финальная фильтрация по SNR (после деноизинга)
df_clean = df_clean[df_clean['snr_db'] >= MIN_SNR_AFTER_DENOISE]

# Сортировка для читаемости
df_clean = df_clean.sort_values(['speaker_id', 'utterance_id']).reset_index(drop=True)

df_clean.to_csv(MANIFEST_OUT, index=False, encoding='utf-8')
print(f'✅ Манифест сохранён: {MANIFEST_OUT}')
print(f'   Записей: {len(df_clean)}')
print(f'   Общая длит.: {df_clean["duration_sec"].sum()/60:.1f} мин')

In [ ]:
# ── Шаг 7: Отчёт о качестве после обработки ─────────────────

print('=' * 60)
print('  ОТЧЁТ О КАЧЕСТВЕ ПОСЛЕ ДЕНОИЗИНГА')
print('=' * 60)

# SNR до vs после
df_denoised = df_clean[df_clean['was_denoised']]
df_clean_only = df_clean[~df_clean['was_denoised']]

print(f'\nЗаписей без деноизинга (SNR изначально ≥ 20 дБ): {len(df_clean_only)}')
print(f'Записей с деноизингом: {len(df_denoised)}')

if len(df_denoised) > 0:
    snr_improvement = df_denoised['snr_db'] - df_denoised['snr_db_original']
    print(f'\nУлучшение SNR после DeepFilterNet3:')
    print(f'  Среднее улучшение: +{snr_improvement.mean():.1f} дБ')
    print(f'  Медиана:           +{snr_improvement.median():.1f} дБ')
    print(f'  Мин/Макс:          {snr_improvement.min():.1f} / {snr_improvement.max():.1f} дБ')

# Распределение SNR после
print(f'\nРаспределение SNR (после обработки):')
bins = [0, 10, 15, 20, 25, 100]
labels = ['<10', '10–15', '15–20', '20–25', '>25']
df_clean['snr_bin'] = pd.cut(df_clean['snr_db'], bins=bins, labels=labels)
for label, count in df_clean['snr_bin'].value_counts().sort_index().items():
    bar = '█' * (count // 100)
    print(f'  {label:>6} дБ: {count:>5}  {bar}')

# По дикторам
print(f'\nПо дикторам:')
spk_stats = (
    df_clean.groupby('speaker_id')['duration_sec']
    .agg(['count', 'sum'])
    .assign(total_min=lambda x: (x['sum'] / 60).round(1))
    .sort_values('total_min', ascending=False)
)
for sp, row in spk_stats.iterrows():
    icon = '✅' if row['total_min'] >= 45 else ('🟡' if row['total_min'] >= 15 else '🔴')
    print(f'  {icon} {sp:<30} {row["count"]:>5} уттер. | {row["total_min"]:>6.1f} мин')

print(f'\n✅ Корпус готов к обучению StarGANv2-VC!')
print(f'   manifest_clean.csv: {MANIFEST_OUT}')
print(f'   Аудио:              {OUT_DIR}/')

In [ ]:
# ── Шаг 8 (Опционально): Прослушивание примеров ─────────────
# Сравниваем исходное и деноизированное аудио

from IPython.display import Audio, display
import random

# Берём случайную обработанную запись
sample = df_clean[df_clean['was_denoised']].sample(1).iloc[0]
clean_path = str(Path(OUT_DIR) / sample['rel_path'])
orig_path  = str(Path(CORPUS_DIR) / sample['rel_path'])

# Находим исходный файл (до деноизинга)
# Учтём что имя могло измениться из-за сегментации
base_id = sample['utterance_id'].split('_seg')[0]
orig_path_try = str(Path(CORPUS_DIR) / sample['speaker_id'] / f'{base_id}.wav')

print(f'Пример: {sample["utterance_id"]}')
print(f'Диктор: {sample["speaker_id"]}')
print(f'SNR до:  {sample["snr_db_original"]:.1f} дБ')
print(f'SNR после: {sample["snr_db"]:.1f} дБ')
print(f'Транскрипт: {sample["transcript"]}')

if os.path.exists(orig_path_try):
    print('\n▶ Оригинал:')
    display(Audio(orig_path_try))

print('\n▶ После деноизинга:')
display(Audio(clean_path))

In [ ]:
# ── Шаг 9: Генерация train/val/test списков для StarGANv2-VC ─

COLAB_BASE = OUT_DIR  # абсолютный путь для Colab

for split in ['train', 'val', 'test']:
    split_df = df_clean[df_clean['split'] == split]
    list_path = Path(OUT_DIR) / f'{split}_list.txt'
    
    with open(str(list_path), 'w', encoding='utf-8') as f:
        for _, row in split_df.iterrows():
            abs_path = f'{COLAB_BASE}/{row["rel_path"]}'
            f.write(f'{abs_path}|{row["domain_id"]}\n')
    
    print(f'  {split}_list.txt: {len(split_df)} записей')

# Статистика по доменам в train
train_df = df_clean[df_clean['split'] == 'train']
print(f'\nДомены в train:')
print(train_df.groupby(['domain_id', 'speaker_id'])['utterance_id'].count().to_string())

print(f'\n✅ Списки сохранены в {OUT_DIR}')